[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ddumu/dourado-minguell-eml-mia-um-p1/blob/main/Entornos_Complejos/MonteCarloTodasLasVisitas.ipynb)

# **Monte Carlo con Políticas epsilon-soft**

_Esto es un ejemplo de uso de Gymnasium e informe sobre un experimento de aprendizaje por refuerzo_

````
Luis D. Hernández.
<ldaniel at um.es>
````

Este notebook describe un experimento de aprendizaje por refuerzo utilizando el algoritmo de Monte Carlo con políticas epsilon-soft. El propósito de este análisis es entrenar un agente en un entorno de gym con el juego "FrozenLake", un entorno estándar en el que el agente debe aprender a moverse a través de un mapa en busca de una meta, evitando caer en agujeros. A continuación, se presenta una descripción de las diferentes partes del código y el proceso utilizado en el experimento.

🎯 **Ojo, el código debe modificarse para ser un auténtico Monte Carlo. Supongo que sabrás darte cuenta.**

## **1. Preparación del Entorno**

La preparación consta de las siguientes partes:
- **Instalación de Dependencias**: Se instalan las librerías necesarias para utilizar el entorno `gymnasium` para la simulación, con el objetivo de crear un ambiente controlado para que el agente pueda interactuar.
- **Importación de Librerías**: Se importan las bibliotecas necesarias como `numpy` para el manejo de matrices y `matplotlib` para la visualización de los resultados.

- **Importación del Entorno "FrozenLake"**:
Se cargan dos versiones del entorno "FrozenLake": una de 4x4 y otra de 8x8. Ambas versiones no son resbaladizas, lo que facilita la comprensión de los resultados, dado que el entorno resbaladizo podría dificultar la comprensión inicial del aprendizaje.

#### 3. **Funciones para Mostrar los Resultados**
   - Se define una función para graficar la proporción de recompensas obtenidas en cada episodio del entrenamiento. Esto ayuda a visualizar el progreso del agente en términos de su desempeño durante el entrenamiento.



##### _________ **Código de la Instalación e Importación**
----

In [1]:
%%capture
#@title Instalamos gym
!pip install 'gym[box2d]==0.20.0'

## Instalación de algunos paquetes.
#!apt-get update
## Para usar gymnasium[box2d]
#!apt install swig
#!pip install gymnasium[box2d]

In [ ]:
#@title Importamos librerias
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
import gymnasium as gym

In [ ]:
#@title Importamos el lago helado
name = 'FrozenLake-v1'
env4 = gym.make(name, is_slippery=False, map_name="4x4", render_mode="ansi") # No resbaladizo para entender mejor los resultados.
env8 = gym.make(name, is_slippery=False, map_name="8x8", render_mode="ansi") # No resbaladizo para entender mejor los resultados.

## **2. Diseño del Agente**

El diseño del agente consta de dos partes, el algoritmo con el que aprende y las políticas (toma de decisiones) que realiza.

- **Políticas del Agente**
   - **Política epsilon-soft**: Se define una política donde todas las acciones tienen una probabilidad de ser elegida.
   - **Política epsilon-greedy**: basada en la política epsilon-soft. De esta forma el agente tiene una pequeña probabilidad de explorar (tomar una acción aleatoria) y una mayor probabilidad de explotar (tomar la acción que considera mejor). Esto permite equilibrar la exploración y la explotación.
   - **Política greedy**: Es la usada una vez que "ha aprendido".

- **Algoritmo de Iteración de Valor**
  - Se implementa el algoritmo de iteración de valor utilizando Monte Carlo.
  - Se usa una versión "on-policy" de Monte Carlo con políticas epsilon greedy sobre una política epsilon-soft.
  - Se basa en el criterio de todas las visitas.
  - Otro aspecto es que la actualización de los retornos no se realiza en el orden inverso a las visitas.

#### **Código de las políticas y algoritmo MC**
----------------

In [ ]:
# @title Políticas del agente y Aproximación Lineal para SARSA Semi-Gradiente

# Acciones
LEFT, DOWN, RIGHT, UP = 0, 1, 2, 3

# Función para obtener el vector de características x(s, a) usando codificación one-hot
def get_features(state, action, num_states, nA):
    # Vector de tamaño total: num_states * nA
    x = np.zeros(num_states * nA)
    # Activamos la característica correspondiente al par (estado, acción)
    feature_idx = state * nA + action
    x[feature_idx] = 1.0
    return x

# Función para calcular Q(s, a) a partir de los pesos w
def get_Q_value(w, state, action, num_states, nA):
    x = get_features(state, action, num_states, nA)
    return np.dot(w, x)

# Función para obtener los valores Q de todas las acciones en un estado dado
def get_all_Q_values_for_state(w, state, num_states, nA):
    Q_s = np.zeros(nA)
    for a in range(nA):
        Q_s[a] = get_Q_value(w, state, a, num_states, nA)
    return Q_s

# Política epsilon-soft adaptada a aproximación por función
def random_epsilon_greedy_policy(w, epsilon, state, num_states, nA):
    pi_A = np.ones(nA, dtype=float) * epsilon / nA
    Q_s = get_all_Q_values_for_state(w, state, num_states, nA)
    best_action = np.argmax(Q_s)
    pi_A[best_action] += (1.0 - epsilon)
    return pi_A

# Política epsilon-greedy para la selección de acciones
def epsilon_greedy_policy(w, epsilon, state, num_states, nA):
    pi_A = random_epsilon_greedy_policy(w, epsilon, state, num_states, nA)
    return np.random.choice(np.arange(nA), p=pi_A)

# Política Greedy a partir de los valores Q. Se usa para mostrar la solución.
def pi_star_from_Q(env, Q):
    done = False
    pi_star = np.zeros([env.observation_space.n, env.action_space.n])
    state, info = env.reset() # empezamos arriba a la izquierda = 0
    actions = ""
    while not done:
        # Volvemos a usar la matriz Q directamente como en el cuaderno original
        action = np.argmax(Q[state, :])
        actions += f"{action}, "
        pi_star[state, action] = action
        state, reward, terminated, truncated, info = env.step(action)
        done = terminated or truncated
    return pi_star, actions

In [ ]:
#@title Algoritmo SARSA Semi-gradiente

def semi_gradient_sarsa(env, num_episodes=5000, epsilon=0.4, decay=False, discount_factor=1.0, alpha=0.1):
  num_states = env.observation_space.n
  nA = env.action_space.n

  # Inicializamos el vector de pesos w de forma aleatoria o en ceros
  # Tamaño: (num_estados * num_acciones)
  w = np.zeros(num_states * nA)

  # Métricas para las gráficas
  stats = 0.0
  list_stats = [stats]
  list_episodes_length = []
  step_display = num_episodes / 10

  for t in tqdm(range(num_episodes)):
      state, info = env.reset(seed=100)
      done = False
      episode_reward = 0.0
      step_count = 0

      if decay:
          epsilon = min(1.0, 1000.0 / (t + 1))

      # 1. Seleccionar la acción inicial A mediante política epsilon-greedy
      action = epsilon_greedy_policy(w, epsilon, state, num_states, nA)

      while not done:
          step_count += 1

          # 2. Ejecutar la acción A, observar R y el nuevo estado S'
          new_state, reward, terminated, truncated, info = env.step(action)
          done = terminated or truncated
          episode_reward += reward # Acumulamos para estadísticas sin descuento para evaluar éxito

          # Calcular x(s, a) y q(s, a, w) actuales
          x_current = get_features(state, action, num_states, nA)
          q_current = np.dot(w, x_current)

          if done:
              # Si es un estado terminal, el valor del siguiente estado-acción es 0
              target = reward
          else:
              # 3. Seleccionar la siguiente acción A' usando la política basada en w
              new_action = epsilon_greedy_policy(w, epsilon, new_state, num_states, nA)

              # Calcular q(s', a', w)
              q_next = get_Q_value(w, new_state, new_action, num_states, nA)
              target = reward + discount_factor * q_next

          # 4. Actualización Semi-gradiente de los pesos:
          # w <- w + alpha * [Target - q(s, a, w)] * Gradiente
          # En aproximación lineal, Gradiente = x(s, a)
          w += alpha * (target - q_current) * x_current

          # Actualizar estado y acción para la siguiente iteración del bucle
          if not done:
              state = new_state
              action = new_action

      # Guardamos datos sobre la evolución (proporción acumulada de éxito)
      stats += episode_reward
      list_stats.append(stats / (t + 1))
      list_episodes_length.append(step_count)

      # Mostrar progreso
      if t % step_display == 0 and t != 0:
          print(f"success rate: {stats / t:.4f}, epsilon: {epsilon:.4f}")

  # Reconstruimos una matriz Q final ficticia solo para mantener la compatibilidad
  # con los prints del cuaderno original.
  Q_final = np.zeros([num_states, nA])
  for s in range(num_states):
      for a in range(nA):
          Q_final[s, a] = get_Q_value(w, s, a, num_states, nA)

  return w, Q_final, list_stats, list_episodes_length

In [ ]:
import numpy as np

# Aseguramos que las funciones necesarias existan en el contexto actual
def test_sarsa_update():
    # Setup: 2 estados, 2 acciones
    num_states = 2
    nA = 2
    w = np.zeros(num_states * nA)
    alpha = 0.1
    gamma = 0.9

    state = 0
    action = 0
    reward = 1.0
    next_state = 1
    next_action = 1

    # q(s, a) inicial es 0 usando la función del notebook
    x_curr = get_features(state, action, num_states, nA)
    q_curr = np.dot(w, x_curr)

    # Simulamos un q(s', a') ya aprendido de 0.5
    # Forzamos un peso para s=1, a=1
    w[next_state * nA + next_action] = 0.5
    q_next = get_Q_value(w, next_state, next_action, num_states, nA)

    target = reward + gamma * q_next
    error = target - q_curr

    # Actualización
    w_old = w.copy()
    w += alpha * error * x_curr

    print(f"Target calculado: {target}")
    print(f"Error TD: {error}")
    print(f"Peso actualizado para (s=0, a=0): {w[0]}")

    assert w[0] > w_old[0], "El peso debería haber aumentado para una recompensa positiva"
    print("\n¡La lógica de actualización matemática de su SARSA Semi-gradiente es correcta!")

# Ejecutamos el test
test_sarsa_update()

## **3. Experimentación**

   - En esta sección, el algoritmo de Monte Carlo con la política epsilon (decaimiento) se ejecuta tanto para el entorno de 4x4 como al de 8x8 de FrozenLake sin resbalar.
   
   - En ambos casos se realiza un entrenamiento con un número determinado de episodios (5000 en concreto)

   - Además en el escenario 8x8 el  epsilon tiene decaimiento de acuerdo a la expresión: $\epsilon = min(1.0, 1000.0/(t+1))$

   - Durante el entrenamiento hay una visualización de la proporción de recompensas obtenidas a lo largo del entrenamiento.

   - Junto a dicho volcado se muestra gráficamente la proporcion de recompensas obtendias.

   - También se hace un volcado de los valores Q de cada estado, donde se muestra cómo el agente valora diferentes acciones en distintos estados del entorno, lo que puede interpretarse como su conocimiento sobre las mejores estrategias para alcanzar la meta sin caer en los agujeros.

   - Además, se muestra la política óptima derivada de los valores Q. Esta política es la que el agente seguiría si tuviera que elegir siempre la acción que maximiza su recompensa esperada.

   

### **3.1 Repressentaciones Gráficas**

Para comprobar el aprendizaje se mostrará la función $f(t)=\frac{\sum_{i=1}^t R_i}{t}$ para $t=1,2,\ldots, NumeroEpisodios$. La justificación es la siguiente. Como sabemmos que el retorno en el estados inicial 1 (pues no hay descuento) o 9, si se divide por el número de episodios ejecutados se calcular el porcentaje de recompensas positivas obtenidas. Dicho de otra forma, nos dirá el porcentaje de veces que el agente ha llegado al estado terminal.

*TODO:* Contruir una gráfica que muestre la longitud de los episodios en cada estado junto con la curva de tendencia.

In [ ]:
# @title Funciones para mostrar los resultados

def plot(list_stats):
  # Creamos una lista de índices para el eje x
  indices = list(range(len(list_stats)))

  # Creamos el gráfico
  plt.figure(figsize=(6, 3))
  plt.plot(indices, list_stats)

  # Añadimos título y etiquetas
  plt.title('Proporción de recompensas')
  plt.xlabel('Episodio')
  plt.ylabel('Proporción')

  # Mostramos el gráfico
  plt.grid(True)
  plt.show()

# Define la función para mostrar el tamaño de los episodios
# Pon aquí tu código.
def plot_episodes_length(list_episodes_length):
  # Creamos una lista de índices para el eje x
  indices = list(range(len(list_episodes_length)))

  # Creamos el gráfico
  plt.figure(figsize=(6, 3))
  plt.plot(indices, list_episodes_length)

  # Añadimos título y etiquetas
  plt.title('Tamaño de los Episodios')
  plt.xlabel('Episodio')
  plt.ylabel('Tamaño')

  # Mostramos el gráfico
  plt.grid(True)
  plt.show()

### **3.2 Experimentación en el escenario 4x4**



   - Se realizan 5000 epsisodios y se actualizan los valores Q (valor de acción) basándose en las recompensas obtenidas durante cada episodio completo (e.d. aplicamos Monte Carlo) Se apica una política $\epsilon$ greedy sobre una política $\epsilon$ soft con un valor $\epsilon$ constante




In [ ]:
# @title Aprendizaje (SARSA Semi-gradiente 4x4)
# Agregamos alpha=0.1. Nota que ahora la función devuelve también 'w'
w4, Q, list_stats, list_episodes_length = semi_gradient_sarsa(
    env4, num_episodes=50000, epsilon=0.4, discount_factor=0.99, alpha=0.1
)


In [ ]:
#@title Proporción de aciertos por número de episodios

plot(list_stats)
print(f"Máxima proporcion: {list_stats[-1]}")

In [ ]:
#@title Evolución del tamaño por episodio

plot_episodes_length(list_episodes_length)
print(f"Máximo tamaño de episodio:", max(list_episodes_length))

####.
Mostramos los valores Q para cada estado. Cada estado tienen 4 valores, que se corresponden con las 4 acciones que se pueden en cada estado.

In [ ]:
# @title Política final 4x4
LEFT, DOWN, RIGHT, UP = 0,1,2,3
pi, actions = pi_star_from_Q(env4, Q)

print("Política óptima obtenida\n", pi, f"\n Acciones {actions} \n Para el siguiente grid\n", env4.render())

- También se muestra la política óptima (greedy) obtenida a partir del aprendizaje anterior.

- Cada estado tienen 4 valores, pero todos son 0 menos 1. Es decir, en cada estado se aplica de manera determinística una única acción.

*TODO:* Mostrar de forma gráfica el escenario.

In [ ]:
# @title Política final
LEFT, DOWN, RIGHT, UP = 0,1,2,3
pi, actions = pi_star_from_Q(env4, Q)

print("Política óptima obtenida\n", pi, f"\n Acciones {actions} \n Para el siguiente grid\n", env4.render())
print()

### **3.3 Experimentación en el escenario 8x8**

  - Se realizan 5000 epsisodios y se actualizan los valores Q (valor de acción) basándose en las recompensas obtenidas durante cada episodio completo (e.d. aplicamos Monte Carlo) Se apica una política $\epsilon$ greedy sobre una política $\epsilon$ soft con un valor $\epsilon$ decreciente



In [ ]:
# @title Aprendizaje (SARSA Semi-gradiente 8x8) - Ajustado para Convergencia
# Aumentamos el número de episodios y suavizamos el aprendizaje
w8, Q, list_stats, list_episodes_length = semi_gradient_sarsa(
    env8,
    num_episodes=100000,
    epsilon=1.0,
    decay=True,
    discount_factor=0.99,
    alpha=0.01
)

# Nota: En semi_gradient_sarsa, podrías considerar cambiar la línea de decay a:
# epsilon = max(0.1, 10000.0 / (t + 1))
# para asegurar que siempre haya un mínimo de exploración.

In [ ]:
#@title Proporción de aciertos por número de episodios

plot(list_stats)
print(f"Máxima proporcion: {list_stats[-1]}")

In [ ]:
#@title Evolución del tamaño por episodio

plot_episodes_length(list_episodes_length)
print(f"Máximo tamaño de episodio:", max(list_episodes_length))

####.
Mostramos los valores Q para cada estado. Cada estado tienen 4 valores, que se corresponden con las 4 acciones que se pueden en cada estado.

In [ ]:
# @title Política final 8x8
LEFT, DOWN, RIGHT, UP = 0,1,2,3
pi, actions = pi_star_from_Q(env8, Q)

print("Política óptima obtenida\n", pi, f"\n Acciones {actions} \n Para el siguiente grid\n", env8.render())

- También se muestra la política óptima (greedy) obtenida a partir del aprendizaje anterior.

- Cada estado tienen 4 valores, pero todos son 0 menos 1. Es decir, en cada estado se aplica de manera determinística una única acción.

*TODO:* Mostrar de forma gráfica el escenario.

In [ ]:
# @title Política final
LEFT, DOWN, RIGHT, UP = 0,1,2,3
pi, actions = pi_star_from_Q(env8, Q)

print("Política óptima obtenida\n", pi, f"\n Acciones {actions} \n Para el siguiente grid\n", env8.render())
print()

## **4. Análisis y Estudios Futuros**

### **4.1 Análisis de Resultados**

- En los dos entornos (4x4 y 8x8), el agente comienza con un conocimiento muy limitado, pero gradualmente mejora su desempeño a medida que avanza en los episodios. Este comportamiento se puede observar en el gráfico de la proporción de recompensas, que aumenta con el tiempo.
- En el entorno 4x4, la máxima proporción de éxito alcanzada fue 0.522, mientras que en el entorno 8x8, la máxima alcanzada fue 0.914. Esto refleja que el agente aprendió a optimizar su estrategia en un entorno más complejo.
- La política óptima obtenida muestra las acciones recomendadas por el agente en cada estado del entorno. En el entorno 8x8, la política es más compleja debido a la mayor cantidad de estados y la dificultad del entorno.

### **4.2 Propuestas para Estudios Futuros**

1. **Evaluar con Otros Entornos**: Sería interesante aplicar este algoritmo a otros entornos más complejos de `gym`, como "Taxi-v3" o "MountainCar", para analizar cómo se comporta el agente en situaciones con dinámicas más complicadas.
   
2. **Optimización del Decaimiento de Epsilon**: Aunque se utilizó un decaimiento de epsilon en el segundo experimento, se podría investigar la efectividad de diferentes tasas de decaimiento o incluso explorar algoritmos como `Q-learning` para comparar su desempeño. Graficamente se trataría de mostrar la curva de la tasa de aciertos para distintas funciones de decaimientos

3. **Análisis del Impacto de los descuentos en las Recompensas**: El estudio se ha hecho para $\gamma = 1$; pero no se ha probado qué pasa cuando  $0 \leq \gamma < 1$. Se trataría de estudiar la curva para distintos valores de $\gamma$

4. **Nuevas gráficas**: Aquí solo se ha usado la proporción de aciertos, pero sería interesante qué relación entre dicha tasa y las tamaños de los episodios.

4. **Ampliación del Algoritmo**: Explorar otros enfoques de Monte Carlo o incluso combinar Monte Carlo con otros algoritmos de aprendizaje por refuerzo, como el Deep Q-Network (DQN), podría mejorar aún más los resultados en entornos más complejos.
